# Feature Engineering

Input: `science_sample_balanced.pkl` (raw balanced sample of science vs. general subreddit posts)

Output: `data_final_features.pkl` (cleaned dataset with all engineered features: semantic similarity, readability, compression ratio, profanity, sentiment, subjectivity, entity retention, pronoun rate)

In [1]:
# Install / verify all required packages
# %pip install -q sentence-transformers textstat better-profanity vaderSentiment textblob spacy seaborn scikit-learn statsmodels
!python -m spacy download en_core_web_sm --quiet
print("All packages ready.")

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
All packages ready.


In [2]:
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import textstat
from better_profanity import profanity
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from textblob import TextBlob
import spacy

c:\Users\ludwi\anaconda3\envs\docana_project\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load Data

In [3]:
data_final = pd.read_pickle("../data/science_sample_balanced.pkl")
print(data_final.shape)
data_final.head()

(83664, 13)


,Index,author,body,normalizedBody,content,content_len,summary,summary_len,id,subreddit,subreddit_id,title,is_science
0,90,Occamsrazor1,Every basic ecology class teaches this. The id...,Every basic ecology class teaches this. The id...,Every basic ecology class teaches this. The id...,112,We need to stop being such selfish bastards,8,c6hj5o3,science,t5_mouw,NaN,True
1,91,Suddenfury,i was thinking about exactly this just yesterd...,i was thinking about exactly this just yesterd...,i was thinking about exactly this just yesterd...,148,why are the fishfarm industry so small scale?,8,c6hml12,science,t5_mouw,NaN,True
2,104,flutterbug32,One thing I have been told multiple times is t...,One thing I have been told multiple times is t...,One thing I have been told multiple times is t...,270,who you know is more helpful sometimes than wh...,11,c6rgfy8,ecology,t5_2qoii,NaN,True
3,226,champyonfiyah,I'll take a crack at this. Starting with: ...,I'll take a crack at this. Starting with: ...,I'll take a crack at this. Starting with: ...,213,Yes you can do that with an arduino.,8,c6v4cup,arduino,t5_2qknj,NaN,True
4,300,Destructor1701,The *apparent* size - which is a function of s...,The apparent size - which is a function of s...,The apparent size - which is a function of s...,97,Distance matters only as part of the function ...,24,c71qcs8,space,t5_2qh87,NaN,True


## Semantic Similarity (SBERT Cosine Similarity)

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6890.49it/s]


In [5]:
content_embeddings = model.encode(
    data_final["content"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

summary_embeddings = model.encode(
    data_final["summary"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

eps = 1e-8  # avoid division by zero

data_final["cosine_similarity"] = (
    np.sum(content_embeddings * summary_embeddings, axis=1)
    / (
        np.linalg.norm(content_embeddings, axis=1)
        * np.linalg.norm(summary_embeddings, axis=1)
        + eps
    )
)

data_final["cosine_similarity"].describe()

Batches: 100%|██████████| 2615/2615 [10:49<00:00,  4.03it/s]


count    83664.000000
mean         0.474890
std          0.200700
min         -0.179651
25%          0.351054
50%          0.505636
75%          0.626213
max          0.944736
Name: cosine_similarity, dtype: float64

## Readability Metric

In [6]:
def compute_readability(text):
    """Compute multiple readability scores for a given text."""
    return {
        'flesch_reading_ease': textstat.flesch_reading_ease(text),
        'flesch_kincaid_grade': textstat.flesch_kincaid_grade(text),
        'gunning_fog': textstat.gunning_fog(text),
        'coleman_liau': textstat.coleman_liau_index(text)
    }

content_readability = data_final['content'].apply(compute_readability)
summary_readability = data_final['summary'].apply(compute_readability)

content_readability_df = pd.DataFrame(content_readability.tolist()).add_prefix('content_')
summary_readability_df = pd.DataFrame(summary_readability.tolist()).add_prefix('summary_')

data_final = pd.concat([data_final, content_readability_df, summary_readability_df], axis=1)

readability_metrics = ['flesch_reading_ease', 'flesch_kincaid_grade', 'gunning_fog', 'coleman_liau']
for metric in readability_metrics:
    data_final[f'delta_{metric}'] = (
        data_final[f'summary_{metric}'] - data_final[f'content_{metric}']
    )

## Cleaning Data

In [7]:
n_before = len(data_final)

# 1. Remove duplicates
data_final = data_final.drop_duplicates(subset='content')

# 2. Filter implausible readability scores on content side (lists, markup artifacts)
data_final = data_final[data_final['content_flesch_kincaid_grade'] < 30]

# 3. Filter summaries that are too short to be meaningful
data_final = data_final[data_final['summary_len'] > 3]

n_after = len(data_final)
print(f"Removed {n_before - n_after} observations ({(n_before - n_after) / n_before * 100:.1f}%)")
print(f"Remaining: {n_after}")

Removed 4431 observations (5.3%)
Remaining: 79233


In [8]:
# Non-alphabetic character ratio (lists, markup artifacts, etc.)
data_final['content_alpha_ratio'] = data_final['content'].apply(
    lambda x: sum(c.isalpha() for c in x) / len(x) if len(x) > 0 else 0
)
data_final = data_final[data_final['content_alpha_ratio'] >= 0.5]
print(f"Remaining: {len(data_final)}")

Remaining: 79179


In [9]:
print(data_final['is_science'].value_counts())
print(data_final['is_science'].value_counts(normalize=True).round(3))

is_science
False    39999
True     39180
Name: count, dtype: int64
is_science
False    0.505
True     0.495
Name: proportion, dtype: float64


## Compression Ratio

In [10]:
data_final['compression_ratio'] = data_final['summary_len'] / data_final['content_len']
data_final['compression_ratio'].describe()

count    79179.000000
mean         0.156691
std          0.155484
min          0.001427
25%          0.059375
50%          0.107595
75%          0.191358
max          0.997319
Name: compression_ratio, dtype: float64

## Profanity Metric

In [11]:
data_final['content'] = data_final['content'].astype(str)
data_final['summary'] = data_final['summary'].astype(str)

clean_wordset = [str(w) for w in profanity.CENSOR_WORDSET]
profanity_pattern = r'\b(' + '|'.join(re.escape(w) for w in clean_wordset) + r')\b'

data_final['content_profanity'] = data_final['content'].str.lower().str.contains(
    profanity_pattern, regex=True, na=False).astype(int)
data_final['summary_profanity'] = data_final['summary'].str.lower().str.contains(
    profanity_pattern, regex=True, na=False).astype(int)

data_final['delta_profanity'] = data_final['summary_profanity'] - data_final['content_profanity']

C:\Users\ludwi\AppData\Local\Temp\ipykernel_29932\2959557296.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  data_final['content_profanity'] = data_final['content'].str.lower().str.contains(
C:\Users\ludwi\AppData\Local\Temp\ipykernel_29932\2959557296.py:9: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  data_final['summary_profanity'] = data_final['summary'].str.lower().str.contains(


## Sentiment and Subjectivity

In [12]:
analyzer = SentimentIntensityAnalyzer()

data_final['content_sentiment'] = data_final['content'].apply(
    lambda x: analyzer.polarity_scores(x)['compound'])
data_final['summary_sentiment'] = data_final['summary'].apply(
    lambda x: analyzer.polarity_scores(x)['compound'])

data_final['content_subjectivity'] = data_final['content'].apply(
    lambda x: TextBlob(x).sentiment.subjectivity)
data_final['summary_subjectivity'] = data_final['summary'].apply(
    lambda x: TextBlob(x).sentiment.subjectivity)

data_final['delta_sentiment'] = data_final['summary_sentiment'] - data_final['content_sentiment']
data_final['delta_subjectivity'] = data_final['summary_subjectivity'] - data_final['content_subjectivity']

## Entity Retention Rate

In [13]:
nlp = spacy.load('en_core_web_sm')

def entity_retention_rate(content, summary):
    content_entities = set(ent.text.lower() for ent in nlp(content).ents)
    summary_entities = set(ent.text.lower() for ent in nlp(summary).ents)

    if len(content_entities) == 0:
        return None

    retained = content_entities.intersection(summary_entities)
    return len(retained) / len(content_entities)

data_final['entity_retention_rate'] = data_final.apply(
    lambda row: entity_retention_rate(row['content'], row['summary']), axis=1)

data_final['entity_retention_rate'] = data_final['entity_retention_rate'].fillna(
    data_final['entity_retention_rate'].median()
)

## Pronoun Usage

In [14]:
# Pronoun usage as a register feature
# Personal pronouns signal narrative/interpersonal vs. formal/impersonal writing
pronoun_pattern = re.compile(
    r'\b(i|me|my|mine|myself|'
    r'we|us|our|ours|ourselves|'
    r'you|your|yours|yourself|yourselves|'
    r'he|him|his|she|her|hers|'
    r'they|them|their|theirs|themselves|'
    r'himself|herself)\b',
    re.IGNORECASE
)
word_pattern = re.compile(r'\b[a-zA-Z]+\b')

def pronoun_rate(text):
    words = word_pattern.findall(str(text))
    if not words:
        return 0.0
    return len(pronoun_pattern.findall(str(text))) / len(words)

data_final['content_pronoun_rate'] = data_final['content'].apply(pronoun_rate)
data_final['summary_pronoun_rate'] = data_final['summary'].apply(pronoun_rate)
data_final['delta_pronoun_rate'] = data_final['summary_pronoun_rate'] - data_final['content_pronoun_rate']

## Save

In [15]:
data_final.to_pickle('../data/data_final_features.pkl')

print(f"Saved {len(data_final)} rows and {len(data_final.columns)} columns")
print("\nColumns saved:")
print(data_final.columns.tolist())

Saved 79179 rows and 41 columns

Columns saved:
['Index', 'author', 'body', 'normalizedBody', 'content', 'content_len', 'summary', 'summary_len', 'id', 'subreddit', 'subreddit_id', 'title', 'is_science', 'cosine_similarity', 'content_flesch_reading_ease', 'content_flesch_kincaid_grade', 'content_gunning_fog', 'content_coleman_liau', 'summary_flesch_reading_ease', 'summary_flesch_kincaid_grade', 'summary_gunning_fog', 'summary_coleman_liau', 'delta_flesch_reading_ease', 'delta_flesch_kincaid_grade', 'delta_gunning_fog', 'delta_coleman_liau', 'content_alpha_ratio', 'compression_ratio', 'content_profanity', 'summary_profanity', 'delta_profanity', 'content_sentiment', 'summary_sentiment', 'content_subjectivity', 'summary_subjectivity', 'delta_sentiment', 'delta_subjectivity', 'entity_retention_rate', 'content_pronoun_rate', 'summary_pronoun_rate', 'delta_pronoun_rate']
